# Gold Processing — Fact View Main

Creates `ecommerce.gold.fact_view_main`, a wide analytical view joining all gold tables built so far:

* **Base** — `gold_order_items` (183,378 rows, all 21 columns)
* **Date** — `gold_date` joined on `date_id` (derived from the order's `dt` as `yyyyMMdd` int)
* **Product** — `gold_product` joined on `product_id` (sku, category/brand codes and names, attributes, rating_count)
* **Customer** — `gold_customer` joined on `customer_id` (phone, country/state, region)

Notes:

* Left joins — the fact row always survives even if a dimension is missing (102 known orphan customers → their customer columns are null)
* Every table's columns are included; duplicated metadata names are prefixed by dimension (`date_source_file`, `product_ingested_at`, etc.)
* This is a **view**, not a materialized table — it recomputes on query and always reflects the latest gold tables

In [0]:
# Create the wide fact view joining all gold tables
spark.sql("""
CREATE OR REPLACE VIEW ecommerce.gold.fact_view_main AS
SELECT
    -- order items (fact — all columns)
    oi.*,
    -- date dimension (joined on date_id derived from dt)
    d.date_id,
    d.date,
    d.year,
    d.month_name,
    d.day_name,
    d.is_weekend,
    d.quarter,
    d.week_of_year,
    d.source_file AS date_source_file,
    d.ingested_at AS date_ingested_at,
    -- product dimension (joined on product_id)
    p.sku,
    p.category_code,
    p.category_name,
    p.brand_code,
    p.brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.source_file AS product_source_file,
    p.ingested_at AS product_ingested_at,
    -- customer dimension (joined on customer_id)
    c.phone,
    c.country_code,
    c.country,
    c.state,
    c.region,
    c.source_file AS customer_source_file,
    c.ingested_at AS customer_ingested_at
FROM ecommerce.gold.gold_order_items oi
LEFT JOIN ecommerce.gold.gold_date d
    ON CAST(date_format(oi.dt, 'yyyyMMdd') AS INT) = d.date_id
LEFT JOIN ecommerce.gold.gold_product p
    ON oi.product_id = p.product_id
LEFT JOIN ecommerce.gold.gold_customer c
    ON oi.customer_id = c.customer_id
""")

print(f"View created: ecommerce.gold.fact_view_main")

In [0]:
# Verify the view
spark.sql("""
    SELECT
      COUNT(*) AS rows,
      COUNT_IF(date_id IS NULL) AS unjoined_dates,
      COUNT_IF(sku IS NULL) AS unjoined_products,
      COUNT_IF(region IS NULL) AS unjoined_customers,
      COUNT(DISTINCT order_id) AS distinct_orders
    FROM ecommerce.gold.fact_view_main
""").show()

# Quick sanity: top regions by INR sales straight off the view
spark.sql("""
    SELECT region, ROUND(SUM(sales_amt_INR), 2) AS sales_inr, COUNT(*) AS items
    FROM ecommerce.gold.fact_view_main
    GROUP BY region
    ORDER BY sales_inr DESC
""").show()

display(spark.table("ecommerce.gold.fact_view_main").limit(5))